In [1]:
# Load Dataset

import pandas as pd
import numpy as np

df=pd.read_csv("./dados_streaming.csv")
df

,id_cliente,idade,mensalidade,meses_contrato,suporte_tecnico,target
0,1000,56.0,17.60,12,Não,Fiel
1,1001,69.0,12.06,44,Não,Fiel
2,1002,46.0,16.13,36,Não,Cancelou
3,1003,32.0,10.75,4,Não,Cancelou
4,1004,60.0,10.42,5,Não,Fiel
...,...,...,...,...,...,...
495,1495,71.0,15.76,32,Sim,Cancelou
496,1496,60.0,14.96,29,Não,Fiel
497,1497,54.0,16.01,3,Não,Fiel
498,1498,50.0,10.08,40,Sim,Cancelou


# Phase 1: Data Audit

## 1.1. Cleaning: How many customers are missing the age? Fill in these values with the median age.

In [4]:
print("Null values in the age column: \n", df['idade'].isnull().sum())

Null values in the age column: 
 15


In [44]:
df['idade'] = df['idade'].fillna(df['idade'].median())
print("Null values in the age column after filling: \n", df['idade'].isnull().sum())

Null values in the age column after filling: 
 0


## 1.2. Cardinality: How many unique values are there in column suporte_tecnico? And in the target?

In [13]:
print ("Unique values in suporte_tecnico: \n", df['suporte_tecnico'].unique())

Unique values in suporte_tecnico: 
 ['Não' 'Sim']


In [14]:
print ("Unique values in target: \n", df['target'].unique())

Unique values in target: 
 ['Fiel' 'Cancelou']


## 1.3. Frequency: What percentage of customers have "Canceled" the service? (Tip: use value_counts(normalize=True)).

In [ ]:
dict(df['target'].value_counts(normalize=True)).get('Cancelou')*100 # Just to get the percentage of customers that have "Cancelou"

np.float64(28.999999999999996)

# Phase 2: Filters and Logic

## 2.1 Segmentation: Create a new DataFrame called clientes_premium that contains only those who pay a monthly fee greater than 15.00 and have been in the service for more than 12 months.

In [20]:
clientes_premium = df[(df['mensalidade'] > 15.00) & (df['meses_contrato'] > 12)]
clientes_premium.head()

,id_cliente,idade,mensalidade,meses_contrato,suporte_tecnico,target
2,1002,46.000000,16.13,36,Não,Cancelou
5,1005,25.000000,15.21,37,Sim,Fiel
7,1007,56.000000,16.27,42,Sim,Fiel
8,1008,46.195876,17.16,28,Sim,Fiel
9,1009,40.000000,19.66,31,Sim,Fiel


## 2.2. Removal: Does the id_cliente column help predict user behavior? If not, remove it from the DataFrame.

In [23]:
df = df.drop(columns=['id_cliente'])
df.head()

,idade,mensalidade,meses_contrato,suporte_tecnico,target
0,56.0,17.60,12,Não,Fiel
1,69.0,12.06,44,Não,Fiel
2,46.0,16.13,36,Não,Cancelou
3,32.0,10.75,4,Não,Cancelou
4,60.0,10.42,5,Não,Fiel


# Phase 3: Model Preparation (Encoding)

## 3.1. Manual Mapping: Turn the target column into numbers: True -> 0, Canceled -> 1.

In [34]:
target_mapping = {'Fiel': 0, 'Cancelou': 1}
df['target_values'] = df['target'].map(target_mapping)
print(df[['target', 'target_values']].sample(10))

     target  target_values
375     NaN            NaN
400     NaN            NaN
4       NaN            NaN
495     NaN            NaN
86      NaN            NaN
271     NaN            NaN
382     NaN            NaN
358     NaN            NaN
404     NaN            NaN
136     NaN            NaN


## 3.2. Binary Mapping: Transform column suporte_tecnico into: No -> 0, Yes -> 1.

In [33]:
suporte_mapping = {'Não': 0, 'Sim': 1}
df['suporte_tecnico_values'] = df['suporte_tecnico'].map(suporte_mapping)
print(df[['suporte_tecnico', 'suporte_tecnico_values']].sample(10))

    suporte_tecnico  suporte_tecnico_values
411             Não                       0
82              Não                       0
334             Não                       0
38              Não                       0
207             Não                       0
218             Sim                       1
303             Sim                       1
420             Sim                       1
14              Não                       0
421             Não                       0


# Phase 4: ML Insights

## 4.1. Grouping: What is the average meses_contrato for those who canceled vs. those who are loyal?

In [36]:
grouping_result = df.groupby('target')['meses_contrato'].mean()
print(grouping_result)

target
Cancelou    25.434483
Fiel        24.247887
Name: meses_contrato, dtype: float64


## 4.2. Correlation: Is there any strong relationship between the monthly fee and the numerical target?

In [ ]:
correlation = df['mensalidade'].corr(df['target_values'])
print("Correlation between mensalidade and target_values: ", correlation)

KeyError: 'target_values'

# Phase 5: EXTRA

## Create a new variable that helps the model understand the value of each customer. Create a column named "Total_amount" What you should do:

### Multiply the monthly fee by the number of contract_months.

### Create a business rule: If the Total_amount is greater than 500, the customer is considered "VIP" (1), otherwise it is "Normal" (0).

In [12]:
df['Total_amount'] = df['mensalidade'] * df['meses_contrato']
df['customer_value'] = np.where(df['Total_amount'] > 500, 'VIP', 'Normal')
print(df[['Total_amount', 'customer_value']].sample(10))


     Total_amount customer_value
251        365.64         Normal
236         19.87         Normal
198        334.42         Normal
430        113.00         Normal
108        366.80         Normal
11         380.77         Normal
418        321.36         Normal
440        317.86         Normal
228        822.60            VIP
99          57.63         Normal
